In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

# Load data
df = pd.read_csv("loan_default.csv")

# View first rows
print(df.head())
print(df.dtypes)

   SK_ID_CURR  AGE  ANNUAL_INCOME  LOAN_AMOUNT  EMPLOYMENT_YEARS  \
0      100000   59          77767        58019                22   
1      100001   49          48054        81154                32   
2      100002   35          95319        12373                 7   
3      100003   63         122789        40281                11   
4      100004   28          65397        25187                 9   

   CREDIT_SCORE  EXISTING_LOANS  TARGET  
0           432               2       1  
1           438               1       1  
2           792               4       1  
3           369               4       0  
4           314               0       1  
SK_ID_CURR          int64
AGE                 int64
ANNUAL_INCOME       int64
LOAN_AMOUNT         int64
EMPLOYMENT_YEARS    int64
CREDIT_SCORE        int64
EXISTING_LOANS      int64
TARGET              int64
dtype: object


In [2]:
# Define target
target = 'TARGET'

# Features (drop ID column)
X = df.drop(['SK_ID_CURR', target], axis=1)
y = df[target]

# Split Train/Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Identify numeric columns
numeric_cols = ['AGE', 'ANNUAL_INCOME', 'LOAN_AMOUNT', 'EMPLOYMENT_YEARS', 'CREDIT_SCORE', 'EXISTING_LOANS']

In [3]:
# Fill missing numeric values with median
imputer = SimpleImputer(strategy='median')

X_train_num = pd.DataFrame(imputer.fit_transform(X_train[numeric_cols]),
                           columns=numeric_cols,
                           index=X_train.index)

X_test_num = pd.DataFrame(imputer.transform(X_test[numeric_cols]),
                          columns=numeric_cols,
                          index=X_test.index)


In [4]:
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)

X_train_poly = pd.DataFrame(poly.fit_transform(X_train_num),
                            columns=poly.get_feature_names_out(numeric_cols),
                            index=X_train_num.index)

X_test_poly = pd.DataFrame(poly.transform(X_test_num),
                           columns=poly.get_feature_names_out(numeric_cols),
                           index=X_test_num.index)

In [5]:
for col in ['ANNUAL_INCOME', 'LOAN_AMOUNT', 'CREDIT_SCORE']:
    X_train_poly[col + '_bin'] = pd.qcut(X_train_num[col], q=4, labels=False)
    X_test_poly[col + '_bin'] = pd.qcut(X_test_num[col], q=4, labels=False)\
    

In [6]:
# Baseline with numeric features only
model_baseline = RandomForestClassifier(n_estimators=200, random_state=42)
model_baseline.fit(X_train_num, y_train)

y_pred_baseline = model_baseline.predict(X_test_num)
y_prob_baseline = model_baseline.predict_proba(X_test_num)[:,1]

In [7]:

corr_matrix = X_train_poly.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]
X_train_poly.drop(to_drop, axis=1, inplace=True)
X_test_poly.drop(to_drop, axis=1, inplace=True)

In [8]:
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train_poly, y_train)

importances = pd.Series(rf.feature_importances_, index=X_train_poly.columns)
important_features = importances[importances > 0.01].index  # keep important features

X_train_selected = X_train_poly[important_features]
X_test_selected = X_test_poly[important_features]

In [9]:
rfe = RFE(rf, n_features_to_select=min(20, X_train_selected.shape[1]))
rfe.fit(X_train_selected, y_train)

X_train_rfe = X_train_selected.loc[:, rfe.support_]
X_test_rfe = X_test_selected.loc[:, rfe.support_]

In [10]:
model_fe = RandomForestClassifier(n_estimators=300, random_state=42)
model_fe.fit(X_train_rfe, y_train)

y_pred_fe = model_fe.predict(X_test_rfe)
y_prob_fe = model_fe.predict_proba(X_test_rfe)[:,1]

In [11]:
metrics = {
    "Accuracy": [accuracy_score(y_test, y_pred_baseline), accuracy_score(y_test, y_pred_fe)],
    "F1 Score": [f1_score(y_test, y_pred_baseline), f1_score(y_test, y_pred_fe)],
    "Precision": [precision_score(y_test, y_pred_baseline), precision_score(y_test, y_pred_fe)],
    "Recall": [recall_score(y_test, y_pred_baseline), recall_score(y_test, y_pred_fe)],
    "ROC-AUC": [roc_auc_score(y_test, y_prob_baseline), roc_auc_score(y_test, y_prob_fe)]
}

comparison_df = pd.DataFrame(metrics, index=["Before FE", "After FE"])
print(comparison_df)

           Accuracy  F1 Score  Precision    Recall   ROC-AUC
Before FE     0.609  0.735989   0.637427  0.870607  0.539090
After FE      0.594  0.721536   0.632212  0.840256  0.549591
